# 12. Bérleti Piac, Hozamszámítás és a Neil Smith-féle Rent Gap

**Cél**: A kőbányai bérleti piac (N = 180 db) elemzése, a bruttó hozamok és a Price-to-Rent ráta kiszámítása, valamint a Neil Smith-féle bérleti rés vizsgálata.

---

### 📖 Miért keresett a kőbányai bérleti piac?
Kőbánya a bérbeadási célú lakásbefektetők kedvelt célpontja, mivel a mérsékeltebb vételárak miatt a bérleti díjak arányaiban kedvezőbb megtérülést nyújtanak.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
kiado = df[df['listing_type'] == 'kiado'].copy()
print(f"Adatbázis: {len(elado)} db eladó és {len(kiado)} db kiadó hirdetés.")

### 1. Kettős Hozamszámítás és Megtérülési KPI-k

**📌 Mit mutatnak a fenti KPI kártyák pontos adatai?**
A kőbányai bérleti piac (N = 180 db) és eladási piac (N = 1 140 db) tényleges, számított mutatói:
- **Átlagos Havi Bérleti Díj**: **260 985 Ft/hó** (medián: **250 000 Ft/hó**).
- **Átlagos Havi Fajlagos Bérleti Díj**: **5 351 Ft/m²/hó** (medián: **5 232 Ft/m²/hó**).
- **Kiadó lakások átlagos mérete**: **52.5 m²** (kisebb, mint az eladók 57.6 m²-es átlaga!).
- **Bruttó Bérleti Hozam m² alapon**: **5.25%**!
- **Bruttó Bérleti Hozam egységár alapon**: **4.55%**!
- **Price-to-Rent (P/R) ráta**: **22.0 év**!

**🔍 Miért tér el a fajlagos (5.25%) és az egységár-alapú (4.55%) hozam?**
A bérbeadott lakások átlagos mérete (52.5 m²) érezhetően kisebb, mint az eladó lakásoké (57.6 m²). Mivel a kisebb lakások fajlagos bérleti díja magasabb, a m² alapú számítás 5.25%-os, míg a teljes lakásárral számolt hozam 4.55%-os bruttó megtérülést mutat.
- A **22.0 éves P/R ráta** azt jelenti, hogy átlagosan 22 évnyi bruttó bérleti díjbevétel fedezi az eladási árat.

**💡 Befektetői tanulság**: Kőbánya 4.5 - 5.3% közötti bruttó hozamszintje stabil, kiszámítható készpénzáramot biztosít a befektetőknek.

In [ ]:
atlag_berlet_huf = kiado['price_huf'].mean()
median_berlet_huf = kiado['price_huf'].median()
atlag_berlet_nm = kiado['nm_ar_huf'].mean()
median_elado_nm = elado['nm_ar_huf'].median()
median_elado_ar = elado['price_huf'].median()

# 1. Fajlagos m² alapú hozam
brutto_hozam_pct = (atlag_berlet_nm * 12 / median_elado_nm) * 100
pr_rata_ev = median_elado_nm / (atlag_berlet_nm * 12)

# 2. Egységár alapú hozam
brutto_hozam_egyseg_pct = (atlag_berlet_huf * 12 / median_elado_ar) * 100
pr_rata_egyseg_ev = median_elado_ar / (atlag_berlet_huf * 12)

kpi_cards = [
    ("Átlagos Havi Bérlet", fmt_huf(atlag_berlet_huf) + " / hó", f"Medián: {fmt_huf(median_berlet_huf)}", "#1e3a8a"),
    ("Bérleti Fajlagos Díj", fmt_huf(atlag_berlet_nm) + " / m²", "Havi fajlagos díj", "#2563eb"),
    ("Fajlagos Bruttó Hozam (m²)", f"{brutto_hozam_pct:.2f}%", f"P/R: {pr_rata_ev:.1f} év", "#059669"),
    ("Egységár Bruttó Hozam (lakás)", f"{brutto_hozam_egyseg_pct:.2f}%", f"P/R: {pr_rata_egyseg_ev:.1f} év", "#d97706"),
    ("Kiadó Lakások Aránya", f"{len(kiado)/len(df)*100:.1f}%", f"{len(kiado)} db hirdetés", "#7c3aed"),
    ("Nettó Hozam (85% kihaszn.)", f"{brutto_hozam_pct * 0.85 * 0.85:.2f}%", "Költségek levonása után", "#10b981")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. A Neil Smith-féle Bérleti Rés (Rent Gap) Elemzése

**📌 Mit mutat a Rent Gap vizsgálat?**
A felújítási potenciállal bíró elhanyagolt ingatlanok jelenlegi bérleti értéke és a felújítás után elérhető piaci bérleti díjak közötti rést.
- Városközpontban és Ligetteleken a polgári téglaházak felújításával jelentős bérletidíj-növekmény realizálható.
- Újhegyen a lakótelepi panelek bérleti díjai homogén sávban mozognak.

In [ ]:
agg_elado = elado.groupby('varosresz')['nm_ar_huf'].median().reset_index(name='elado_nm_ar')
agg_kiado = kiado.groupby('varosresz')['nm_ar_huf'].median().reset_index(name='kiado_nm_ar')

merged_yield = pd.merge(agg_elado, agg_kiado, on='varosresz', how='inner')
merged_yield['brutto_hozam_pct'] = (merged_yield['kiado_nm_ar'] * 12 / merged_yield['elado_nm_ar']) * 100
merged_yield['pr_ratio'] = merged_yield['elado_nm_ar'] / (merged_yield['kiado_nm_ar'] * 12)

fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(
    go.Bar(x=merged_yield['varosresz'], y=merged_yield['elado_nm_ar'], name='Eladási Ár/m² (HUF)', marker_color='#2563eb'),
    secondary_y=False
)
fig1.add_trace(
    go.Scatter(x=merged_yield['varosresz'], y=merged_yield['brutto_hozam_pct'], name='Bruttó Bérleti Hozam (%)', mode='lines+markers', line=dict(color='#10b981', width=3), marker=dict(size=10)),
    secondary_y=True
)

fig1.update_layout(
    title_text='Eladási négyzetméterárak és bruttó bérleti hozamok városrészenként',
    template=PLOTLY_TEMPLATE,
    height=450,
    xaxis_tickangle=-30
)
fig1.update_yaxes(title_text='Eladási Ár / m² (HUF)', secondary_y=False)
fig1.update_yaxes(title_text='Bruttó Hozam (%)', secondary_y=True)
fig1.show()

# Hozamtáblázat
display(HTML("<div style='max-width: 700px; margin: 15px 0;'>" + merged_yield.round(2).to_html(classes='table table-bordered table-striped', index=False) + "</div>"))

### 3. Bérleti Díjak és Lakásméret Összefüggése

**📌 Mit mutat a scatter pontdiagram?**
A havi bérleti díj alakulását a lakásméret függvényében:
- Egy 30-40 m²-es kislakás bérleti díja ~180-220 ezer Ft, míg egy 60 m²-es lakásé ~280-320 ezer Ft.
- A bérleti piac a szobaszámot és az önálló életteret értékeli a legmagasabbra.

In [ ]:
# Bérleti rés elemzése állapotonként
allapot_stat = df.groupby(['allapot', 'listing_type'])['nm_ar_huf'].median().unstack()
allapot_stat = allapot_stat.dropna()

if 'kiado' in allapot_stat.columns and 'elado' in allapot_stat.columns:
    allapot_stat['Eves_Berlet_m2'] = allapot_stat['kiado'] * 12
    allapot_stat['Hozam_pct'] = (allapot_stat['Eves_Berlet_m2'] / allapot_stat['elado']) * 100
    allapot_stat['Tokesitett_Ertek'] = allapot_stat['Eves_Berlet_m2'] / 0.05
    
    # Kiszámoljuk a felújítandó állapot és a legmagasabb (felújított/kiváló) állapot tőkésített értéke közötti különbséget (Rent Gap)
    max_potencial = allapot_stat['Tokesitett_Ertek'].max()
    allapot_stat['Rent_Gap'] = max_potencial - allapot_stat['Tokesitett_Ertek']
    
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(
        x=allapot_stat.index,
        y=allapot_stat['Tokesitett_Ertek'],
        name='Aktuális Tőkésített Bérleti Érték',
        marker_color='#2563eb'
    ))
    fig2.add_trace(go.Bar(
        x=allapot_stat.index,
        y=allapot_stat['Rent_Gap'],
        name='Potenciális Rent Gap (Bérleti Rés)',
        marker_color='#ef4444'
    ))
    fig2.update_layout(
        barmode='stack',
        title='Neil Smith-féle Rent Gap (Bérleti Rés) Kőbányán Állapotonként (5% Tőkésítési Rátával)',
        xaxis_title='Műszaki Állapot',
        yaxis_title='Becsült Érték (Ft/m²)',
        template=PLOTLY_TEMPLATE,
        height=450
    )
    fig2.show()

# Kiadó lakások méret vs bérleti díj szórásdiagramja
fig3 = px.scatter(
    kiado,
    x='alapterulet_nm',
    y='price_huf',
    color='varosresz',
    trendline='ols',
    title='Alapterület vs. Havi bérleti díj a kiadó lakások piacán',
    labels={'alapterulet_nm': 'Alapterület (m²)', 'price_huf': 'Bérleti díj (HUF / hó)'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=450)
fig3.show()

### 4. Interaktív Bérleti Hozam és Megtérülés Kalkulátor

**📌 Számolja ki saját befektetésének megtérülését!**
Állítsa be a vételárat, a várt bérleti díjat, a költségeket és az üresedési időt, a kalkulátor pedig azonnal megadja a valós nettó hozamot és a megtérülési időt!

In [ ]:
# Valós idejű Neil Smith Rent Gap és Bérleti Megtérülés Kalkulátor
html_rent_gap = '''
<div id="rent_gap_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">📊 Neil Smith Rent Gap & Bérleti Megtérülés Kalkulátor</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Dinamikus hozamszámítás, felújítási értéknövekmény és járadék-rés realizáció valós időben</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Motor</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap:20px; margin-bottom:20px;">
    <!-- 1. oszlop: Bázis paraméterek -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">1. Bázis Ingatlan és Bérleti Díj</div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Vételár (M Ft):</span> <span id="rg_lbl_vetel" style="color:#2563eb; font-weight:700;">50 M Ft</span>
        </div>
        <input type="range" id="rg_vetel" min="30" max="120" value="50" step="5" style="width:100%; accent-color:#2563eb;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Havi bérleti díj (ezer Ft):</span> <span id="rg_lbl_berlet" style="color:#2563eb; font-weight:700;">250 ezer Ft/hó</span>
        </div>
        <input type="range" id="rg_berlet" min="150" max="500" value="250" step="10" style="width:100%; accent-color:#2563eb;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Kihasználtsági ráta:</span> <span id="rg_lbl_occ" style="color:#059669; font-weight:700;">95%</span>
        </div>
        <input type="range" id="rg_occ" min="70" max="100" value="95" step="5" style="width:100%; accent-color:#059669;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:6px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Üzemeltetés & amortizáció:</span> <span id="rg_lbl_cost" style="color:#dc2626; font-weight:700;">15%</span>
        </div>
        <input type="range" id="rg_cost" min="5" max="30" value="15" step="5" style="width:100%; accent-color:#dc2626;" oninput="recalcRentGap()">
      </div>
    </div>

    <!-- 2. oszlop: Felújítás és Rent Gap realizáció -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">2. Rent Gap Értéknövelő Beruházás</div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Felújítási tőkeráfordítás (M Ft):</span> <span id="rg_lbl_felujitas" style="color:#7c3aed; font-weight:700;">5.0 M Ft</span>
        </div>
        <input type="range" id="rg_felujitas" min="0" max="25" value="5" step="1" style="width:100%; accent-color:#7c3aed;" oninput="recalcRentGap()">
      </div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Bérletnövekedés felújítás után:</span> <span id="rg_lbl_novek" style="color:#059669; font-weight:700;">+30%</span>
        </div>
        <input type="range" id="rg_novek" min="0" max="60" value="30" step="5" style="width:100%; accent-color:#059669;" oninput="recalcRentGap()">
      </div>

      <div style="background:#eff6ff; border-left:4px solid #2563eb; padding:10px 12px; border-radius:4px; font-size:12px; color:#1e40af; margin-top:10px;">
        🏢 <b>Neil Smith tézis:</b> A járadék-rés (Rent Gap) a felújítás nélküli aktuális tőkésített bérleti érték és a legmagasabb minőségű (potenciális) tőkésített érték közötti különbség.
      </div>
    </div>
  </div>

  <!-- KPI Kártyák -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:10px;">
    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Bruttó Bérleti Hozam</div>
      <div id="rg_res_brutto" style="font-size:26px; font-weight:800; color:#1d4ed8; margin:4px 0;">--%</div>
      <div style="font-size:11px; color:#2563eb;">Kínálati vételárra vetítve</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Tiszta Nettó Hozam</div>
      <div id="rg_res_netto" style="font-size:26px; font-weight:800; color:#15803d; margin:4px 0;">--%</div>
      <div style="font-size:11px; color:#16a34a;">Üresedés & fenntartás után</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Valós Megtérülés</div>
      <div id="rg_res_payback" style="font-size:24px; font-weight:800; color:#b45309; margin:4px 0;">-- év</div>
      <div style="font-size:11px; color:#d97706;">Nettó cash flow alapján</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">Realizálható Rent Gap</div>
      <div id="rg_res_gap" style="font-size:24px; font-weight:800; color:#7e22ce; margin:4px 0;">-- M Ft</div>
      <div style="font-size:11px; color:#9333ea;">Tőkésített többletérték</div>
    </div>
  </div>
</div>

<script>
(function() {
  function update() {
    const vetel = parseFloat(document.getElementById('rg_vetel').value) * 1e6;
    const berlet = parseFloat(document.getElementById('rg_berlet').value) * 1e3;
    const occ = parseFloat(document.getElementById('rg_occ').value) / 100;
    const cost = parseFloat(document.getElementById('rg_cost').value) / 100;
    const felujitas = parseFloat(document.getElementById('rg_felujitas').value) * 1e6;
    const novek = parseFloat(document.getElementById('rg_novek').value) / 100;

    document.getElementById('rg_lbl_vetel').innerText = (vetel / 1e6).toFixed(0) + ' M Ft';
    document.getElementById('rg_lbl_berlet').innerText = (berlet / 1e3).toFixed(0) + ' ezer Ft/hó';
    document.getElementById('rg_lbl_occ').innerText = Math.round(occ * 100) + '%';
    document.getElementById('rg_lbl_cost').innerText = Math.round(cost * 100) + '%';
    document.getElementById('rg_lbl_felujitas').innerText = (felujitas / 1e6).toFixed(1) + ' M Ft';
    document.getElementById('rg_lbl_novek').innerText = '+' + Math.round(novek * 100) + '%';

    const eves_brutto = berlet * 12;
    const brutto_h = (eves_brutto / vetel) * 100;
    const netto_eves = (eves_brutto * occ) * (1.0 - cost);
    const netto_h = (netto_eves / vetel) * 100;
    const megterules = netto_eves > 0 ? vetel / netto_eves : 0;

    // Rent gap
    const uj_berlet = berlet * (1.0 + novek);
    const uj_netto_eves = (uj_berlet * 12 * occ) * (1.0 - cost);
    const cap_rate = Math.max(netto_h / 100, 0.04);
    const uj_kapitalizalt = uj_netto_eves / cap_rate;
    const realizalt_gap = (uj_kapitalizalt - vetel - felujitas) / 1e6;

    document.getElementById('rg_res_brutto').innerText = brutto_h.toFixed(2) + '%';
    document.getElementById('rg_res_netto').innerText = netto_h.toFixed(2) + '%';
    document.getElementById('rg_res_payback').innerText = megterules.toFixed(1) + ' év';
    document.getElementById('rg_res_gap').innerText = (realizalt_gap > 0 ? '+' : '') + realizalt_gap.toFixed(1) + ' M Ft';
  }

  window.recalcRentGap = update;
  setTimeout(update, 50);
})();
</script>
'''
display(HTML(html_rent_gap))